# Cycle 1 — Preprocessing

**Dataset:** `premier_league_matches.csv`  

This notebook takes the raw `premier_league_matches.csv` and prepares it for machine learning. Every issue identified during exploration is addressed here step by step. The output is a clean, model-ready dataset saved to `data/processed/`.

In [22]:
import sys, os  


_here = os.getcwd()                        
while not os.path.isdir(os.path.join(_here, 'data')): 
    _p = os.path.dirname(_here)                       
    if _p == _here: raise RuntimeError('project root not found') 
    _here = _p
if _here not in sys.path:
    sys.path.insert(0, _here)                             

from config import Paths, ensure_dirs  
ensure_dirs()  


In [23]:
# LOADING raw data 
import pandas as pd  

df = pd.read_csv(str(Paths.PL_MATCHES_RAW))  # load raw CSV from config-defined path
print('Shape:', df.shape)                    # confirm expected (6840, 40)
df.head()                                    # preview first 5 rows


Shape: (6840, 40)


,Unnamed: 0,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTGS,ATGS,HTGC,...,HTLossStreak3,HTLossStreak5,ATWinStreak3,ATWinStreak5,ATLossStreak3,ATLossStreak5,HTGD,ATGD,DiffPts,DiffFormPts
0,0,19/08/00,Charlton,Man City,4,0,H,0,0,0,...,0,0,0,0,0,0,0.0,0.0,0.0,0.0
1,1,19/08/00,Chelsea,West Ham,4,2,H,0,0,0,...,0,0,0,0,0,0,0.0,0.0,0.0,0.0
2,2,19/08/00,Coventry,Middlesbrough,1,3,NH,0,0,0,...,0,0,0,0,0,0,0.0,0.0,0.0,0.0
3,3,19/08/00,Derby,Southampton,2,2,NH,0,0,0,...,0,0,0,0,0,0,0.0,0.0,0.0,0.0
4,4,19/08/00,Leeds,Everton,2,0,H,0,0,0,...,0,0,0,0,0,0,0.0,0.0,0.0,0.0


### Drop Useless Columns

In [24]:
cols_to_drop = ['Unnamed: 0', 'HTFormPtsStr', 'ATFormPtsStr']  # redundant: row index + string form duplicates
df = df.drop(columns=cols_to_drop)                              # remove all three in one call

print('Shape after dropping useless columns:', df.shape)        # confirm 3 fewer columns: (6840, 37)
print('Remaining columns:', df.columns.tolist())                # list all remaining columns


Shape after dropping useless columns: (6840, 37)
Remaining columns: ['Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR', 'HTGS', 'ATGS', 'HTGC', 'ATGC', 'HTP', 'ATP', 'HM1', 'HM2', 'HM3', 'HM4', 'HM5', 'AM1', 'AM2', 'AM3', 'AM4', 'AM5', 'MW', 'HTFormPts', 'ATFormPts', 'HTWinStreak3', 'HTWinStreak5', 'HTLossStreak3', 'HTLossStreak5', 'ATWinStreak3', 'ATWinStreak5', 'ATLossStreak3', 'ATLossStreak5', 'HTGD', 'ATGD', 'DiffPts', 'DiffFormPts']


### Reconstruct the Target Variable (FTR) Full Time Result

This is the most critical fix in the entire preprocessing pipeline. The original `FTR` column only had two values — Home Win and Not Home 
- `FTHG > FTAG` → H (Home Win)
- `FTHG < FTAG` → A (Away Win)
- `FTHG == FTAG` → D (Draw)

In [25]:
df['FTR'] = df.apply(                                          # apply a row-wise function to reconstruct FTR
    lambda row: 'H' if row['FTHG'] > row['FTAG']              # home scored more → Home Win
    else ('A' if row['FTHG'] < row['FTAG'] else 'D'),          # away scored more → Away Win; equal → Draw
    axis=1                                                       # axis=1 = apply across columns per row
)

print('Reconstructed FTR distribution:')
print(df['FTR'].value_counts())  # verify counts for H, A, D classes


Reconstructed FTR distribution:
FTR
H    3176
A    1913
D    1751
Name: count, dtype: int64


- We now have all 3 correct classes
- Home wins are the most common (3,176 — 46.4%) — confirms home advantage
- Away wins: 1,913 (28.0%)
- Draws: 1,751 (25.6%)
- Total: 3176 + 1913 + 1751 = 6840 ✓ — no rows lost

### Drop Goal Columns (Data Leakage)

- Removing `FTHG` (Full Time Home Goals) and `FTAG` (Full Time Away Goals) from the dataset.
- These columns were only kept temporarily to reconstruct `FTR`. Now that the target is fixed, they must be dropped. Keeping them would be data leakage — the model would learn to predict the result directly from the scoreline, which isn't availbe before the match starts

In [26]:
df = df.drop(columns=['FTHG', 'FTAG'])  # remove goal columns — they reveal the result and would leak the target

print('Shape after dropping goal columns:', df.shape)  # confirm 2 fewer columns: (6840, 35)


Shape after dropping goal columns: (6840, 35)


### Encode Form Columns (HM1–HM5, AM1–AM5)

Converts the last 5 match result columns from string values (W/D/L/M) to numbers.Machine learning models cannot work with string values — every feature must be numeric.

**Encoding used:**
- `W` = 3 (a win earns 3 points in football)
- `D` = 1 (a draw earns 1 point)
- `L` = 0 (a loss earns 0 points)
- `M` = 0 (no previous match — treated as neutral, same as a loss)

In [27]:
result_map = {'W': 3, 'D': 1, 'L': 0, 'M': 0}                              # football point values: W=3, D=1, L=0; M (missing) = 0
form_cols = ['HM1', 'HM2', 'HM3', 'HM4', 'HM5', 'AM1', 'AM2', 'AM3', 'AM4', 'AM5']  # last 5 results per team

for col in form_cols:            # iterate over all 10 form columns
    df[col] = df[col].map(result_map)  # replace string value with integer equivalent

print('Form columns after encoding:') 
print(df[form_cols].head()) # early rows should all be 0 (Matchweek 1 — no prior results)
print(df[form_cols].tail())    
print()
print('Any NaN introduced?', df[form_cols].isnull().sum().sum())  # 0 = all W/D/L/M values mapped successfully


Form columns after encoding:
   HM1  HM2  HM3  HM4  HM5  AM1  AM2  AM3  AM4  AM5
0    0    0    0    0    0    0    0    0    0    0
1    0    0    0    0    0    0    0    0    0    0
2    0    0    0    0    0    0    0    0    0    0
3    0    0    0    0    0    0    0    0    0    0
4    0    0    0    0    0    0    0    0    0    0
      HM1  HM2  HM3  HM4  HM5  AM1  AM2  AM3  AM4  AM5
6835    0    0    0    0    3    1    3    3    3    3
6836    3    1    3    1    0    3    1    3    3    3
6837    0    0    0    0    1    0    1    1    1    0
6838    3    0    3    1    0    3    0    0    1    0
6839    1    3    0    0    1    1    3    3    1    1

Any NaN introduced? 0


### Encode Team Names

 ML models require numeric input. Team names must be converted to numbers. The same encoding (label) is applied to both columns so that, for example, Arsenal = 0 in both `HomeTeam` and `AwayTeam`.

In [28]:
all_teams = pd.concat([df['HomeTeam'], df['AwayTeam']]).unique()  # collect every unique team name from both columns
team_map = {team: idx for idx, team in enumerate(sorted(all_teams))}  # sort alphabetically, assign integer index

df['HomeTeam'] = df['HomeTeam'].map(team_map)  # replace home team string with its integer code
df['AwayTeam'] = df['AwayTeam'].map(team_map)  # same encoding — same integer means same club in both columns

print('Number of unique teams:', len(team_map))  # 44 clubs across 18 seasons (promotion/relegation)
print()
print('Team encoding (first 10):')
print(dict(list(team_map.items())[:10]))         # preview alphabetical integer mapping
print()
print('HomeTeam and AwayTeam after encoding:')
print(df[['HomeTeam', 'AwayTeam']].head())       # confirm columns are now integers


Number of unique teams: 44

Team encoding (first 10):
{'Arsenal': 0, 'Aston Villa': 1, 'Birmingham': 2, 'Blackburn': 3, 'Blackpool': 4, 'Bolton': 5, 'Bournemouth': 6, 'Bradford': 7, 'Brighton': 8, 'Burnley': 9}

HomeTeam and AwayTeam after encoding:
   HomeTeam  AwayTeam
0        11        24
1        12        41
2        13        27
3        15        34
4        21        16


### Extract Season from Date, Then Drop Date

Parses the `Date` column into a proper datetime, extracts the season year as a new feature, then drops the raw date.

**Season logic I use:** If month >= 8 (August onwards) → season = that year. If month < 8 (before August) → season = year - 1. So a match in May 2018 belongs to the 2017 season (2017/18).

In [29]:
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)   # parse 'DD/MM/YY' string to datetime
df['Season'] = df['Date'].apply(                           # derive season year from match date
    lambda x: x.year if x.month >= 8 else x.year - 1      # Aug+ → this year; before Aug → prior year (season spans calendar years)
)

print('Date range:', df['Date'].min(), 'to', df['Date'].max())  # confirm full span from 2000 to 2018
print()
print('Seasons covered:', sorted(df['Season'].unique()))         # all 18 seasons
print('Total seasons:', df['Season'].nunique())

df = df.drop(columns=['Date'])  # drop raw date — season integer is the only temporal feature we need
print()
print('Shape after dropping Date:', df.shape)


Date range: 2000-08-19 00:00:00 to 2018-05-13 00:00:00

Seasons covered: [np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017)]
Total seasons: 18

Shape after dropping Date: (6840, 35)


/var/folders/68/xgnlh9p55dv401kmhh4znb0h0000gn/T/ipykernel_41064/1208080291.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)   # parse 'DD/MM/YY' string to datetime


The dataset covers 18 complete Premier League seasons: 2000/01 through 2017/18. 18 seasons × 380 matches per season = 6,840 rows — this confirms the dataset is complete with no missing seasons. The last match is 13th May 2018 (end of the 2017/18 season). Season is now a numeric feature ranging from 2000 to 2017

### Convert Matchweek to Integer

In [30]:
print('MW dtype before:', df['MW'].dtype)  # expect float64 (artifact from original data merge)
df['MW'] = df['MW'].astype(int)             # convert to integer — matchweek is always a whole number (1–38)
print('MW dtype after:', df['MW'].dtype)    # confirm int64
print('MW range:', df['MW'].min(), 'to', df['MW'].max())  # should span 1–38 for a full PL season


MW dtype before: float64
MW dtype after: int64
MW range: 1 to 38


Matchweek correctly ranges from 1 to 38 (a full Premier League season). No data loss from the conversion, all values were already whole numbers stored as floats

### Encode the Target Variable 

Converting the `FTR` target column from string labels (H/D/A) to numeric values.
- `H` = 2 (Home Win)
- `D` = 1 (Draw)
- `A` = 0 (Away Win)

In [35]:
target_map = {'H': 2, 'D': 1, 'A': 0}  # natural ordering from home team perspective: Away Win=0, Draw=1, Home Win=2
df['FTR'] = df['FTR'].map(target_map)   # replace string labels with integers in-place

print('FTR after encoding:')
print(df['FTR'].value_counts().sort_index())      # counts per class: 0 (away), 1 (draw), 2 (home)
print()
print('2=Home Win, 1=Draw, 0=Away Win')


FTR after encoding:
Series([], Name: count, dtype: int64)

2=Home Win, 1=Draw, 0=Away Win


### Final Data Check

- Before producing/saving the dataset. I Verifythe final state of the dataset — shape, column names, data types, missing values, and a preview of the data, to catch any mistakes if present 

In [32]:
print('Final shape:', df.shape)                   # confirm (6840, 35) — no rows lost, correct column count
print()
print('Final columns:', df.columns.tolist())       # list all 35 column names
print()
print('Data types:')
print(df.dtypes)                                   # all must be numeric — no strings should remain
print()
print('Missing values:', df.isnull().sum().sum())  # must be 0
print()
df.head()                                          # visual sanity check of first 5 rows


Final shape: (6840, 35)

Final columns: ['HomeTeam', 'AwayTeam', 'FTR', 'HTGS', 'ATGS', 'HTGC', 'ATGC', 'HTP', 'ATP', 'HM1', 'HM2', 'HM3', 'HM4', 'HM5', 'AM1', 'AM2', 'AM3', 'AM4', 'AM5', 'MW', 'HTFormPts', 'ATFormPts', 'HTWinStreak3', 'HTWinStreak5', 'HTLossStreak3', 'HTLossStreak5', 'ATWinStreak3', 'ATWinStreak5', 'ATLossStreak3', 'ATLossStreak5', 'HTGD', 'ATGD', 'DiffPts', 'DiffFormPts', 'Season']

Data types:
HomeTeam           int64
AwayTeam           int64
FTR                int64
HTGS               int64
ATGS               int64
HTGC               int64
ATGC               int64
HTP              float64
ATP              float64
HM1                int64
HM2                int64
HM3                int64
HM4                int64
HM5                int64
AM1                int64
AM2                int64
AM3                int64
AM4                int64
AM5                int64
MW                 int64
HTFormPts          int64
ATFormPts          int64
HTWinStreak3       int64
HTWinStr

,HomeTeam,AwayTeam,FTR,HTGS,ATGS,HTGC,ATGC,HTP,ATP,HM1,...,HTLossStreak5,ATWinStreak3,ATWinStreak5,ATLossStreak3,ATLossStreak5,HTGD,ATGD,DiffPts,DiffFormPts,Season
0,11,24,2,0,0,0,0,0.0,0.0,0,...,0,0,0,0,0,0.0,0.0,0.0,0.0,2000
1,12,41,2,0,0,0,0,0.0,0.0,0,...,0,0,0,0,0,0.0,0.0,0.0,0.0,2000
2,13,27,0,0,0,0,0,0.0,0.0,0,...,0,0,0,0,0,0.0,0.0,0.0,0.0,2000
3,15,34,1,0,0,0,0,0.0,0.0,0,...,0,0,0,0,0,0.0,0.0,0.0,0.0,2000
4,21,16,2,0,0,0,0,0.0,0.0,0,...,0,0,0,0,0,0.0,0.0,0.0,0.0,2000


- **6,840 rows preserved** — no data was lost during preprocessing
- **35 columns**: 1 target (`FTR`) + 34 features
- **0 missing values** — clean dataset
- All columns are now numeric — no string values remain
- The dataset is ready for feature engineering or direct model training

In [36]:
# Saving the final processed dataset 
df.to_csv(str(Paths.PL_MATCHES_PROCESSED), index=False)  # index=False avoids writing a pandas row index column
print('Saved to data/processed/premier_league_matches_processed.csv')
print('Shape:', df.shape)  # confirm saved shape matches final state


Saved to data/processed/premier_league_matches_processed.csv
Shape: (6840, 35)
